# 02 - Define the evaluation and weekly call policy

This notebook locks the evaluation rules before any model is fitted. It defines which labels may be used for training, how the Monday standing queue works, how bookings are prioritized, and which metrics and return assumptions will be reported.

It does **not** fit a model, inspect model results, run duplicate sensitivity, or write a modified data file. The reusable calculations live in `modeling/evaluation_policy.py`; this notebook explains the decisions and demonstrates that shared code.

## Operating scenario and evaluation choices

The illustrative operating scenario uses booking-time risk scores, a standing weekly queue, capacity for 150 calls per week, and a $50 property-credit offer. These are case-study assumptions that would need to be replaced with actual operational inputs for a deployment.

The evaluation uses the complete 180-day population and 60/13/13 booking-week periods from notebook 01; one call at most per booking; and a fresh queue at the start of validation and holdout. Cancellation risk and booking value define the core ranking candidates, with proximity to arrival tested as an additional priority term.

Cancellation loss is valued from the actual cancellation date under two explicit resale bounds. That valuation is used only in evaluation and never in a ranking. The 180-day limit remains a population-scope decision and is not an assumed resale window.

Exact matching rows remain in the analysis. Without a real reservation ID, they cannot safely be declared duplicate bookings, so no deduplicated sensitivity analysis is planned.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "hotel_bookings.csv").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "modeling"))

import evaluation_policy as evaluation
from prepare_data import (
    RAW_DATA_PATH, complete_180_rows, model_features, prepare_data,
)

pd.set_option("display.max_columns", 60)

## 1. Reuse the fixed population and chronological periods

Notebook 01 preserves all source rows. Evaluation uses only its complete booking cohorts with lead time from 0 through 180 days. The 86 complete Monday-Sunday booking weeks remain divided into 60 training, 13 validation, and 13 holdout weeks. This notebook does not redefine those periods after seeing results.

In [2]:
raw = pd.read_csv(RAW_DATA_PATH)
prepared = prepare_data(raw)
modeling_population = complete_180_rows(prepared)

period_summary = (
    modeling_population.groupby("booking_period_split")
    .agg(
        rows=("source_row_id", "size"),
        booking_weeks=("booking_week_start", "nunique"),
        first_booking_week=("booking_week_start", "min"),
        last_booking_week=("booking_week_start", "max"),
    )
    .reindex(["train", "validation", "holdout"])
)
display(period_summary)

validation_start = period_summary.loc["validation", "first_booking_week"]
holdout_start = period_summary.loc["holdout", "first_booking_week"]
print("Modeling rows:", f"{len(modeling_population):,}")
print("Validation starts:", validation_start.date())
print("Holdout starts:", holdout_start.date())

,rows,booking_weeks,first_booking_week,last_booking_week
booking_period_split,,,,
train,48738,60,2015-07-06,2016-08-22
validation,11050,13,2016-08-29,2016-11-21
holdout,17075,13,2016-11-28,2017-02-20


Modeling rows: 76,863
Validation starts: 2016-08-29
Holdout starts: 2016-11-28


## 2. Allow only known outcomes into model fitting

The booking-period split and the training-label rule answer different questions:

- `booking_period_split == 'train'` says the booking was made during one of the 60 training weeks.
- `arrival_date < validation_start` says its cancellation or no-show outcome would already have been resolved when validation begins.

A training-period booking with a later arrival remains in the prepared data, but its future label is not allowed into this model fit. The rule uses arrival date for every outcome rather than admitting early cancellations through `reservation_status_date`; doing the latter would include unresolved future bookings only when they happened to cancel early. Strictly `<`, rather than `<=`, avoids assuming that a same-day no-show was known at the start of the day.

In [3]:
nominal_train = modeling_population.loc[
    modeling_population["booking_period_split"].eq("train")
].copy()
fit_mask = nominal_train["arrival_date"].lt(validation_start)
model_fit_population = nominal_train.loc[fit_mask].copy()

display(
    pd.Series(
        {
            "nominal training-period rows": len(nominal_train),
            "rows available for model fitting": len(model_fit_population),
            "training-period rows not yet resolved": (~fit_mask).sum(),
        },
        name="rows",
    ).to_frame()
)

X_fit = model_features(model_fit_population)
y_fit = model_fit_population["is_canceled"].copy()
print(
    f"Feature matrix reserved for later models: {X_fit.shape[0]:,} rows "
    f"and {X_fit.shape[1]} features"
)
print("Target rows reserved for later models:", f"{len(y_fit):,}")

,rows
nominal training-period rows,48738
rows available for model fitting,43264
training-period rows not yet resolved,5474


Feature matrix reserved for later models: 43,264 rows and 27 features
Target rows reserved for later models: 43,264


## 3. Keep the booking-time prediction fixed

The later logistic-regression and CatBoost notebooks will fit all preprocessing and model parameters only on `model_fit_population`. Each validation or holdout booking will receive one out-of-sample `p_cancel` based only on the approved booking-time features. That probability is stored and never recalculated while the booking waits in the queue.

Validation will be used to choose the model and policy. The chosen specification will then be frozen for holdout; it will not be refitted or retuned using holdout outcomes. This is a retrospective chronological cohort experiment, not a perfect replay of a live launch: some validation outcomes are resolved only after holdout bookings were made. A fully prospective model-selection backtest would require more calendar history or a long outcome-delay gap that this file cannot support cleanly.

## 4. Separate cancellation probability from business priority

The model estimates cancellation risk. The weekly decision also considers the amount exposed and how close arrival is:

```text
booking_value_proxy = max(adr, 0) * total_nights
proximity_at_call   = clip(1 - days_to_arrival_at_call / urgency_window_days, 0, 1)
urgency_boost       = 1 + 0.5 * proximity_at_call
risk_value_priority = p_cancel * booking_value_proxy
business_priority   = risk_value_priority * urgency_boost
```

Every term above uses only information available when the list is produced. The mild 30-day urgency boost is 1.00 for bookings at least 30 days out, 1.25 at 15 days, and at most 1.50 at arrival. The 14- and 60-day windows remain sensitivity checks. The 180-day modeling scope is unrelated to this urgency rule.

One consequence should be stated before the results rather than discovered in them. Bookings are scored at booking time and enter the queue on the following Monday, so most are called months before arrival and receive a boost of exactly 1.00. The urgency term can therefore only shift the margin of the list, not reorder it. Notebook 05 measures how much it actually moves rather than assuming the answer.

Negative ADR is floored at zero for this value calculation only; the cleaned ADR is not overwritten. ADR times nights is booked room-revenue exposure, not profit. `business_priority` is a risk/value/urgency priority index, not an adjusted probability or an expected-dollar estimate.

### Ranking-time proximity and evaluation-time resale are different quantities

The boost above uses days to arrival **at the call**, which is knowable when the list is produced. How much revenue a cancellation actually destroys depends on how much notice the hotel received, which is a property of the **cancellation date**. Section 7 values loss that way, and that valuation never enters a ranking score.

An earlier version of this analysis used call-time proximity for both purposes. That was wrong, and wrong in a direction that mattered: it assigned zero loss to every cancellation that happened to be called more than 30 days from arrival, which was over 90 percent of them, so the program's estimated benefit was almost entirely erased by an artifact of when the phone call was placed. A booking called 150 days out that cancels three days before arrival is a near-total loss, not a free one. Keeping the two quantities separate is the correction.

In [4]:
URGENCY_WINDOW_DAYS = evaluation.DEFAULT_URGENCY_WINDOW_DAYS
print("Headline urgency-boost window:", URGENCY_WINDOW_DAYS, "days")

Headline urgency-boost window: 30 days


## 5. Define the Monday standing queue

Validation and holdout are separate fresh-launch simulations. Each starts with an empty queue so training rows cannot enter validation and validation rows cannot enter holdout. This omits any incumbent backlog that a real launch might inherit, which will be reported as a limitation.

Because the proxy booking date has no timestamp, bookings made Monday through Sunday first become callable on the following Monday. On each decision Monday the queue keeps bookings that have arrived in the queue, have not passed their arrival date, are not known to have canceled before Monday, and have never been called. A cancellation dated on Monday remains eligible because the file cannot tell whether it happened before or after that morning's list was produced.

The highest positive priorities are selected, up to the 150-call capacity. Ties use `source_row_id` for reproducibility; it is still only a source-row key, not a real reservation ID. Uncalled rows carry forward, and called rows never return. Each split has 13 decisions. When those decisions end, some active bookings may still be waiting in the queue. They are reported separately rather than giving that split extra call weeks.

In [5]:
policy_population = evaluation.add_first_call_monday(modeling_population)

validation_call_dates = evaluation.call_dates_for_split(
    policy_population, "validation"
)
holdout_call_dates = evaluation.call_dates_for_split(
    policy_population, "holdout"
)
display(
    pd.DataFrame(
        {
            "decision Mondays": [len(validation_call_dates), len(holdout_call_dates)],
            "first call date": [validation_call_dates.min(), holdout_call_dates.min()],
            "last call date": [validation_call_dates.max(), holdout_call_dates.max()],
        },
        index=["validation", "holdout"],
    )
)

,decision Mondays,first call date,last call date
validation,13,2016-09-05,2016-11-28
holdout,13,2016-12-05,2017-02-27


In [6]:
print("Shared queue code: modeling/evaluation_policy.py")
print("Available policy rankings:", ", ".join(evaluation.PRIORITY_COLUMNS))

Shared queue code: modeling/evaluation_policy.py
Available policy rankings: random, risk_only, risk_and_value, risk_value_and_urgency


### Small queue check

This artificial example checks the rules without creating an oracle score from the real target. Booking 1 is called on the first Monday, booking 2 has already canceled, booking 3 enters one week later, and booking 4 has already reached arrival.

In [7]:
toy_bookings = pd.DataFrame(
    {
        "source_row_id": [1, 2, 3, 4],
        "booking_period_split": ["toy"] * 4,
        "booking_week_start": pd.to_datetime(
            ["2024-01-01", "2024-01-01", "2024-01-08", "2024-01-01"]
        ),
        "arrival_date": pd.to_datetime(
            ["2024-01-20", "2024-01-25", "2024-01-30", "2024-01-08"]
        ),
        "reservation_status": ["Check-Out", "Canceled", "Check-Out", "No-Show"],
        "reservation_status_date": pd.to_datetime(
            ["2024-01-22", "2024-01-07", "2024-02-01", "2024-01-08"]
        ),
        "is_canceled": [0, 1, 0, 1],
        "adr": [200.0, 180.0, 150.0, 120.0],
        "total_nights": [2, 3, 2, 1],
        "p_cancel": [0.80, 0.90, 0.70, 0.60],
    }
)
toy_calls, toy_weekly, toy_summary, toy_final_queue = (
    evaluation.simulate_weekly_policy(toy_bookings, "toy", capacity=1)
)

print("Rows selected on each Monday:")
display(
    toy_calls[
        [
            "call_date",
            "source_row_id",
            "risk_value_priority",
            "realized_loss_proxy",
        ]
    ]
)

print("Rows that could or could not enter the queue at their first opportunity:")
display(evaluation.pre_call_attrition(toy_bookings).to_frame())

print("Active queue size and calls made on each Monday:")
display(toy_weekly[["call_date", "eligible_queue", "calls"]])

Rows selected on each Monday:


,call_date,source_row_id,risk_value_priority,realized_loss_proxy
0,2024-01-08,1,320.0,0.0
1,2024-01-15,3,210.0,0.0


Rows that could or could not enter the queue at their first opportunity:


,bookings
canceled before first Monday,1
arrived by first Monday,1
available at first opportunity,2


Active queue size and calls made on each Monday:


,call_date,eligible_queue,calls
0,2024-01-08,1,1
1,2024-01-15,1,1


### Small loss-valuation check

The toy queue above never reaches a cancellation, so it does not exercise the loss valuation. This second check does, on constructed rows, and shows the behaviour that the earlier call-time version got wrong.

Notice is measured from the cancellation date to arrival, so the same booking is valued differently depending on when the guest cancelled, not on when we happened to call. A no-show has zero notice and is a full loss. Under the 30-day resale assumption a cancellation with a month or more of notice is treated as fully recoverable, which is exactly why the no-resale bound is reported alongside it.

In [8]:
loss_example = pd.DataFrame(
    {
        "is_canceled": [1, 1, 1, 1, 1, 0],
        "reservation_status": [
            "No-Show", "Canceled", "Canceled", "Canceled", "Canceled",
            "Check-Out",
        ],
        "arrival_date": pd.to_datetime(["2024-03-01"] * 6),
        "reservation_status_date": pd.to_datetime(
            [
                "2024-03-01",  # no-show, zero notice
                "2024-02-25",  # 5 days notice
                "2024-02-01",  # 29 days notice
                "2024-01-31",  # 30 days notice, at the resale window
                "2023-12-02",  # 90 days notice
                "2024-03-05",  # stayed; checked out after arrival
            ]
        ),
        "booking_value_proxy": [1_000.0] * 6,
    }
)

valued = evaluation.add_realized_loss_columns(loss_example)
display(
    valued[
        [
            "reservation_status",
            "is_canceled",
            "cancellation_notice_days",
            "unrecovered_share_at_cancellation",
            "realized_loss_proxy",
            "full_value_loss_proxy",
        ]
    ].style.format(
        {
            "unrecovered_share_at_cancellation": "{:.2f}",
            "realized_loss_proxy": "{:,.0f}",
            "full_value_loss_proxy": "{:,.0f}",
        },
        na_rep="-",
    )
)

# A booking that stayed contributes nothing to either bound.
assert valued.loc[5, ["realized_loss_proxy", "full_value_loss_proxy"]].eq(0).all()
# A no-show is a full loss under both bounds.
assert valued.loc[0, "realized_loss_proxy"] == valued.loc[0, "full_value_loss_proxy"]
# Notice at or beyond the resale window is fully recoverable under the optimistic bound.
assert valued.loc[[3, 4], "realized_loss_proxy"].eq(0).all()
print("Loss-valuation checks passed.")

,reservation_status,is_canceled,cancellation_notice_days,unrecovered_share_at_cancellation,realized_loss_proxy,full_value_loss_proxy
0,No-Show,1,0.000000,1.00,"1,000","1,000"
1,Canceled,1,5.000000,0.83,833,"1,000"
2,Canceled,1,29.000000,0.03,33,"1,000"
3,Canceled,1,30.000000,0.00,0,"1,000"
4,Canceled,1,90.000000,0.00,0,"1,000"
5,Check-Out,0,-,-,0,0


Loss-valuation checks passed.


## 6. Decide how the models will be compared

All choices are made with validation data. Holdout is used once at the end.

### Choose the cancellation model

First compare logistic regression and CatBoost using cancellation probability only. Each Monday, rank the active, uncalled queue by `p_cancel` and select the top 150. This keeps booking value and the urgency assumption out of the model comparison.

The primary model metric is mean weekly precision@150:

```text
precision@150 = selected bookings that eventually canceled / 150 available call slots
```

The denominator remains 150 if a week does not use every slot, so unused capacity is not hidden. Separately report precision among calls actually made. The better model is the one with higher mean validation precision@150. Also report week-to-week variation. If the models are materially similar, prefer logistic regression because it is easier to explain.

Average precision and Brier score are supporting model checks. ROC AUC may be shown for familiarity, but it is not used to choose the model because it does not represent the weekly call limit. Probability calibration is outside the scope of this ranking comparison; the scores are not treated as calibrated expected-loss estimates.

### Evaluate the calling policy

After choosing the model, use its validation scores to compare three rankings: risk only, risk times booking value, and risk times value times the mild urgency boost. Report for every policy:

- mean weekly precision@150, which counts how many cancellations were reached;
- canceled booking value reached and realized loss reached, which say how expensive those cancellations were;
- average days to arrival at the call, capacity used, and bookings left waiting.

Both dimensions are needed. Precision alone treats every booking as equally costly, while value alone ignores that a cancellation with four months of notice is largely recoverable.

The selection rule, declared before running: prefer the policy that reaches the most realized loss, because protecting revenue is what the program exists to do. If two policies are materially similar on that measure, prefer the simpler ranking, which is the same tiebreak used for the model choice. A policy that gained on loss while collapsing precision@150 would be rejected, and precision is reported for every policy so that trade is visible rather than implied.

Realized loss is an evaluation measure computed from the actual cancellation date. It is never available to the ranking, so a policy cannot be rewarded for a quantity it was allowed to see. The reverse mistake is also avoided: the ranking's urgency term is no longer scored against a loss definition built from that same term.

### Baseline and operational checks

A repeated random ranking shows what the same 150-call queue might achieve without a model. Also report calls used, bookings that became uncallable before their first Monday, and bookings still waiting after the final decision. These explain how the policy behaved but do not replace precision@150 as the primary model metric.

In [9]:
# Later model notebooks use evaluation.model_metric_summary(y_true, p_cancel).

## 7. Estimate return using explicit assumptions

A good call list does not automatically mean the program will make money. The historical data can tell us which selected bookings eventually canceled, roughly how much room revenue was booked, and how much notice the hotel received. It cannot tell us whether the phone call and $50 credit would have prevented those cancellations.

Return is therefore a scenario calculation, not another model metric.

### How a reached cancellation is valued

The business context is that a cancellation hurts when the room cannot be resold, and that resale becomes harder closer to arrival. How much is actually recovered is a business fact this file does not contain, so two bounds are reported instead of one point estimate:

```text
notice_days     = arrival_date - reservation_status_date       (0 for a no-show)
unrecovered     = clip(1 - notice_days / resale_window_days, 0, 1)
realized_loss   = booking_value_proxy * unrecovered            (linear resale recovery)
full_value_loss = booking_value_proxy                          (no resale at all)
```

`realized_loss` with a 30-day resale window is the **optimistic** bound on resale recovery and the **pessimistic** bound on the program's value: it assumes a room canceled a month or more before arrival is resold at full price and costs nothing at all. `full_value_loss` is the **pessimistic** bound on resale recovery and the **optimistic** bound on the program's value: it assumes no cancellation is ever resold. Reality sits between them and depends on occupancy, season, and channel mix. Both use the cancellation date and are evaluation-only, never ranking inputs.

| Input | Meaning | Where it comes from |
|---|---|---|
| Loss reached | Exposure among selected bookings that later canceled, under either bound | Dataset plus the resale assumption |
| Save rate | Share of those outcomes prevented by calling | Assumption; ultimately requires a randomized pilot |
| Contribution margin | Share of protected room revenue that represents economic value | Finance or hotel operations |
| Call cost | Staff cost for each attempted call | Customer operations |
| Effective credit cost per call | Average property-credit cost spread across all attempted calls | Contact, acceptance, redemption, and marginal credit-cost assumptions |

```text
estimated benefit = save rate * contribution margin * loss reached
estimated cost    = calls * (call cost + effective credit cost per call)
net return        = estimated benefit - estimated cost
ROI               = net return / estimated cost
```

The break-even save rate is the smallest share of selected cancellations that must be prevented for benefit to equal cost. Reporting it under both loss bounds is the point of the exercise: it shows how much of the program's business case rests on the resale assumption rather than on model quality.

A 10% save rate assumes the program has the same 10% chance of preventing each selected target-positive outcome, including cancellations and no-shows. Treating outreach as equally effective for both statuses is a simplifying assumption. If effectiveness also varies with booking value, read the rate as an exposure-weighted average. It does **not** mean that the model is 10% accurate.

The grid below uses placeholder save rates and costs to show a range rather than claim one return. An effective credit cost of $25 per attempted call could represent the full $50 being incurred on roughly half of calls. The arithmetic assumes ADR has been converted into the same dollar-equivalent units as the program costs; the source file does not state a currency, and its 2015-2017 amounts are not inflation-adjusted here. These assumptions should be replaced with business inputs and then tested with a randomized pilot.

In [10]:
scenario_grid = pd.MultiIndex.from_product(
    [
        ["realized_loss_proxy", "full_value_loss_proxy"],
        [0.05, 0.10, 0.20],
        [25.0, 50.0],
    ],
    names=[
        "loss_basis",
        "assumed_save_rate",
        "effective_credit_cost_per_call",
    ],
).to_frame(index=False)
scenario_grid["assumed_contribution_margin"] = 0.70
scenario_grid["assumed_call_cost"] = 5.0
display(scenario_grid)

print(
    "Resale window for the optimistic bound:",
    evaluation.DEFAULT_RESALE_WINDOW_DAYS,
    "days",
)
# Later policy results use evaluation.estimate_program_return(..., loss_column=...).

,loss_basis,assumed_save_rate,effective_credit_cost_per_call,assumed_contribution_margin,assumed_call_cost
0,realized_loss_proxy,0.05,25.0,0.7,5.0
1,realized_loss_proxy,0.05,50.0,0.7,5.0
2,realized_loss_proxy,0.10,25.0,0.7,5.0
3,realized_loss_proxy,0.10,50.0,0.7,5.0
4,realized_loss_proxy,0.20,25.0,0.7,5.0
5,realized_loss_proxy,0.20,50.0,0.7,5.0
6,full_value_loss_proxy,0.05,25.0,0.7,5.0
7,full_value_loss_proxy,0.05,50.0,0.7,5.0
8,full_value_loss_proxy,0.10,25.0,0.7,5.0
9,full_value_loss_proxy,0.10,50.0,0.7,5.0


Resale window for the optimistic bound: 30 days


## 8. Handoff to the model notebooks

The next notebooks should now follow one contract:

1. Fit preprocessing and the model only on `model_fit_population`.
2. Produce one immutable booking-time `p_cancel` for each validation and holdout row.
3. Select the model on validation using mean weekly queue precision@150 under the risk-only queue.
4. With the selected model, compare the three calling policies on validation and lock the business policy before holdout, using the rule declared in section 6.
5. Import the shared helpers from `modeling/evaluation_policy.py`; never use `is_canceled` as a score, and never let a cancellation-date quantity into a ranking.
6. Evaluate the frozen model and policy on holdout once, then apply the economic scenario table separately under both loss bounds.

Important limitations remain: `booking_date_proxy` has no timestamp; the source is not a true booking-time snapshot; the queue starts without incumbent backlog; some bookings are still waiting when the 13 decisions end; and a date-only final status cannot resolve same-day ordering. The headline rule therefore keeps cancellations dated on Monday eligible at that Monday's snapshot. Also, `source_row_id` is not a reservation ID; exact matching rows may represent separate bookings; ADR times nights is not profit; the resale recovery curve is an assumption rather than a measured rate; the fixed booking-time probability does not update after a booking survives in the queue; and the data contains no causal evidence that outreach or a $50 credit prevents cancellation.